In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

def prepare():
    module_path = os.path.abspath(os.path.join('..'))
    if module_path not in sys.path:
        sys.path.append(module_path)

In [3]:
import torch
import numpy as np
prepare()

In [4]:
model_params = dict(
    label = "GCN", 
    model = "GCN", 
    normalization = "row_normalization",
    activation = "relu",
    depth = 1,
    regularizer = 0.01,
    pred_method = "svm",
    bias = False,
    alpha_tol = 1e-4,
    solver = "qplayer",
)

certificate_params = dict(
    delta = 0.01,
    TimeLimit = 86400,
    LogToConsole = 1,
    OutputFlag = 1,
    Threads = 2,
    Presolve = 2
)

verbosity_params = dict(
    debug_lvl = "warning"
)  

other_params = dict(
    device = "0",
    dtype = torch.float64,
    allow_tf32 = False,
    path_gurobi_license = "path/to/your/gurobi/license"
)

In [5]:
data_params = dict(
    dataset = "csbm",
    learning_setting = "transductive", 
    specification = dict(
        classes = 2,
        n_trn_labeled = 10,
        n_trn_unlabeled = 0,
        n_val = 10,
        n_test = 180,
        sigma = 1,
        avg_within_class_degree = 1.58 * 2,
        avg_between_class_degree = 0.37 * 2,
        K = 1.5,
        seed = 0 # used to generate the dataset & data split
    )
)

In [6]:
deltas = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]

In [7]:
import pandas as pd

seed = 73211
for delta in deltas:
    certificate_params["delta"] = delta
    from exp_labelcert_binaryclass import run
    result_sample = run(data_params, model_params, certificate_params, verbosity_params, other_params, seed)
    from exp_labelcert_collective import run
    result_collective = run(data_params, model_params, certificate_params, verbosity_params, other_params, seed)
    
    samplewise_robust = result_sample['y_is_robust']
    collective_robust = result_collective['y_is_robust']
    accuracy_samplewise = result_sample['accuracy_cert_pois_robust']
    accuracy_collective = result_collective['accuracy_cert_pois_robust']
    samplewise_testdict = result_sample['test_nodes_info']
    collective_testdict = result_collective['test_nodes_info'] # Same as samplewise
    
    test_feature_norms = samplewise_testdict['test_feature_norms']
    avg_test_degree = samplewise_testdict['avg_test_degree']
    test_feature_dim = samplewise_testdict['test_feature_dim']
    ntk_rank = samplewise_testdict['test_ntk_rank']
    n_test_nodes = samplewise_testdict['n_test_nodes']
    n_test_edges = samplewise_testdict['n_test_edges']
    
    df = pd.DataFrame({
    'node': list(range(len(samplewise_robust))),
    'samplewise_robust': samplewise_robust,
    'collective_robust': collective_robust,
    'feature_norm': test_feature_norms,
    })
    df['accuracy_samplewise'] = accuracy_samplewise
    df['accuracy_collective'] = accuracy_collective
    df['avg_test_degree'] = avg_test_degree
    df['test_feature_dim'] = test_feature_dim
    df['test_ntk_rank'] = ntk_rank
    df['n_test_nodes'] = n_test_nodes
    df['n_test_edges'] = n_test_edges
    
    df.to_csv(f'gcn-{delta:.2f}.csv', index=False)

CSBM mu:
[0.28347334 0.28347334 0.28347334 0.28347334 0.28347334 0.28347334
 0.28347334]
20 alphas found: ['0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100', '0.0100']
Set parameter Username
Set parameter LicenseID to value 2663332
Academic license - for non-commercial use only - expires 2026-05-10
Set parameter BestObjStop to value 0
Set parameter BestBdStop to value 0
Set parameter IntegralityFocus to value 1
Set parameter IntFeasTol to value 0.0001
Set parameter DualReductions to value 0
Set parameter Presolve to value 2
Set parameter Threads to value 2
Set parameter FeasibilityTol to value 0.0001
Set parameter OptimalityTol to value 0.0001
Set parameter TimeLimit to value 86400
Gurobi Optimizer version 11.0.1 build v11.0.1rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: AMD Ryzen 7 4800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thr